In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench 0.5.1.
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade_final")
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066676,0.067000,0.067106,0.067312,0.067791,0.068512,0.069519,0.070839,0.071885
Temperature (°C) [sea_water_potential_temperature]{surface},0.519070,0.519718,0.520606,0.523431,0.528348,0.536228,0.547131,0.559176,0.568834
Salinity (PSU) [sea_water_salinity]{surface},0.579209,0.575225,0.571294,0.567868,0.564335,0.561388,0.558962,0.556696,0.553837
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118513,0.119144,0.120126,0.121542,0.123298,0.125690,0.128619,0.131854,0.134391
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119518,0.120151,0.121217,0.122780,0.125016,0.127787,0.130999,0.134335,0.136917
Temperature (°C) [sea_water_potential_temperature]{50m},0.864040,0.864009,0.863822,0.863773,0.865047,0.867414,0.871056,0.875857,0.878098
Salinity (PSU) [sea_water_salinity]{50m},0.242694,0.242556,0.242353,0.242154,0.242087,0.242168,0.242367,0.242582,0.242491
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112296,0.112781,0.113292,0.113982,0.114900,0.116191,0.117858,0.119719,0.120893
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113248,0.113381,0.113712,0.114297,0.115096,0.116198,0.117799,0.119574,0.120709
Temperature (°C) [sea_water_potential_temperature]{100m},1.058480,1.059019,1.059861,1.061503,1.063658,1.067396,1.072999,1.078135,1.080149


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.810351,42.152386,42.459106,42.693715,42.969119,43.283107,43.660847,44.057347,44.268383


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.116661,0.117119,0.117383,0.118005,0.118991,0.120072,0.122016,0.123887,0.125033
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.122630,0.122884,0.123393,0.123518,0.124531,0.125229,0.127539,0.129853,0.131397


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Temperature (°C) [sea_water_potential_temperature]{surface},0.779878,0.808537,0.777822,0.795134,0.822871,0.864284,0.855822,0.872106,0.900333
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.732588,0.744801,0.764812,0.787385,0.786936,0.804572,0.818360,0.812334,0.810118
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.863915,0.877035,0.861938,0.893189,0.884128,0.892656,0.908484,0.936407,0.941389
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.779499,0.802793,0.792674,0.797075,0.815203,0.813592,0.835739,0.836917,0.870282
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.518621,0.533172,0.528961,0.530821,0.543921,0.560009,0.557884,0.571354,0.590258
Salinity (PSU) [sea_water_salinity]{0-5m},0.257719,0.276822,0.272876,0.301133,0.274720,0.286961,0.269445,0.269581,0.292128
Salinity (PSU) [sea_water_salinity]{5-100m},0.271578,0.267259,0.278630,0.262348,0.301175,0.287139,0.273897,0.287408,0.283422
Salinity (PSU) [sea_water_salinity]{100-300m},0.127370,0.129089,0.130363,0.130556,0.133833,0.133078,0.137462,0.134288,0.137500
Salinity (PSU) [sea_water_salinity]{300-600m},0.081286,0.081876,0.080940,0.081940,0.082924,0.084462,0.084438,0.087469,0.088969
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048436,0.049386,0.050160,0.051710,0.052976,0.054849,0.056522,0.058235,0.060393


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},9.968942,19.276466,28.208366,36.864254,45.313469,53.584652,61.69669


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013589,0.016464,0.019036,0.022333,0.026274,0.030379,0.034720,0.039065,0.042159
Temperature (°C) [sea_water_potential_temperature]{surface},0.175986,0.202640,0.230789,0.260280,0.291510,0.324197,0.359246,0.393193,0.418666
Salinity (PSU) [sea_water_salinity]{surface},0.127545,0.146452,0.162282,0.176640,0.190153,0.205423,0.220888,0.235147,0.246648
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.048775,0.055130,0.061920,0.069655,0.077960,0.086398,0.095320,0.103724,0.109923
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.050472,0.057228,0.064233,0.071835,0.080257,0.089104,0.098038,0.106537,0.112885
Temperature (°C) [sea_water_potential_temperature]{50m},0.305239,0.330217,0.356111,0.385172,0.417936,0.454704,0.493686,0.529302,0.551285
Salinity (PSU) [sea_water_salinity]{50m},0.062948,0.067587,0.073017,0.079096,0.085871,0.093238,0.100913,0.108067,0.112930
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.043765,0.047990,0.053119,0.058843,0.065358,0.072362,0.079860,0.086909,0.091763
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.045776,0.049949,0.054879,0.060536,0.066845,0.073708,0.080977,0.088028,0.093075
Temperature (°C) [sea_water_potential_temperature]{100m},0.272715,0.304555,0.339085,0.378833,0.421831,0.467785,0.515547,0.558994,0.586199


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},31.975681,32.926385,33.709573,34.485893,35.302413,36.160882,37.053452,37.81613,38.470798


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.044166,0.051074,0.058204,0.065617,0.072933,0.080553,0.088143,0.095098,0.099915
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.045946,0.053929,0.062199,0.070200,0.078055,0.086213,0.094047,0.101192,0.105986


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},4.007585,7.94075,12.08022,16.589151,21.528172,26.910919,32.738934
